## LIF Model

Leaky-Integrate-(and)-Fire

The analogy: each neuron is a leaky bucket under a dripping tap.

**Integrate**
- when a connected neighbour fires, it drips some water into that bucket
- how much water per drip depends on weight
- "Integrate" just means adding these drips up over time

**Leaky**
- the bucket has a hole in the bottom so it's leaking out and will empty into nothing (not into other buckets)
- biologically, a neuron doesn't hold charge forever, like this bucket
- decays back to resting state, which is just a charge of 0

**Fire**
- once the water crosses a THRESHOLD, the neuron fires
- each spike delivers weight units to every target
- the bucket is back to being empty, or near empty

simulations need one more ingredient: external input / an artificial nudge
- manually set a neuron's charge to X so that water can circulate in the system

the key thing to observe is how the water distributes:

> "given one initial nudge, how does the charge distribute through a real piece of fly wiring, and does it reach anywhere interesting?"

the initial nudge happens at the eyes, the `ol_sensory` superclass
- this is why the exploration was done first, to know the data and not blindly inject somewhere else
- watch whether it distributes the way predicted in `00_explore.ipynb`

#### Init

In [ ]:
import pandas as pd
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.compute as pc

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

#### Picking a starting neuron

In [ ]:
df = pd.read_feather(neurons_file)

neuron = df.query('superclass == \'ol_sensory\' and status == \'Traced\'').iloc[0]
print(neuron)

neuron_id = neuron['bodyId']
print('bodyId we are interested in: ', neuron_id)

> type is R1-R6
>
> R1-   R6 neurons are specialized for motion detection, image formation, and vision in low-light environments
>
> weird, but somaSide is NaN here, but rootSide is populated with L.
>
> this is the reverse of what was found in the previous notebook. so it seems like if one is empty, check the other
>
> again, this is particularly in a pair due to the _L postfix. to further confirm:

#### Finding its neighbours

next, find its neighbours to build a cluster
- because this only looks at one neuron's direct connections, no batching or duckdb is needed
- pyarrow's filtering happens at the C++ level before things become python objects, hence why this can run without crashing

In [ ]:
dataset = ds.dataset(connections_file, format="feather")

filter_expr = (pc.field("body_pre") == neuron_id) | (pc.field("body_post") == neuron_id)

# only after we have a small subset do we go back to pandas
neighbours = dataset.to_table(filter=filter_expr).to_pandas()
print(neighbours)
print(neighbours.shape)

> 27 of 32 rows have 11139 as body_pre, which makes sense, eyes see then must send that info out
>
> some ids look off, they're in the millions

In [ ]:
# check what superclass these ids are part of
id_to_superclass = df[df['status'] == 'Traced'].set_index('bodyId')['superclass'].to_dict()

neighbours['pre_type'] = neighbours['body_pre'].map(id_to_superclass)
neighbours['post_type'] = neighbours['body_post'].map(id_to_superclass)
print(neighbours)

> every NaN neuron type correlates with either a big/unusual ID and/or weight = 1
>
> those NaN neighbours are connections to segments that exist in the raw EM reconstruction but never got fully proofread into confirmed, typed neurons

In [ ]:
# add a column of pre and post statuses
id_to_status = df.set_index('bodyId')['status'].to_dict()

neighbours['pre_status'] = neighbours['body_pre'].map(id_to_status)
neighbours['post_status'] = neighbours['body_post'].map(id_to_status)
print(neighbours)

> this further confirms the suspicion that the big IDs are not annotated at all

filter out these NaN neighbours since the cluster needed requires neurons that can be labelled and reasoned about

In [ ]:
real_neighbours = neighbours.dropna(subset=['pre_type', 'post_type']) # drop if subset cols has missing values
print(real_neighbours)

#### Setting up the simulation

set up the neurons and connections as plain Python structures

In [ ]:
neuron_ids = real_neighbours[['body_pre', 'body_post']].stack().unique().tolist() # flatten, remove dupes, then list

charge = {nid: 0 for nid in neuron_ids} # dict comprehension -> everyone starts at 0 charge

# convert df rows into tuples -> (pre, post, weight)
# - iterate over specific columns so must filter them
# - index=False excludes index from the tuple, name=None prevents a named tuple
edges = list(real_neighbours[['body_pre', 'body_post', 'weight']].itertuples(index=False, name=None))
print(edges)

#### Parameters

- decay: each tick; whatever charge is sitting in a bucket loses 20% of itself. though fast, it shows change
- threshold: want at least one neuron to actually fire in this toy run, so threshold is kept low
- time_steps: let's see what happens with an arbitrary value
- external_input: at tick 0, the photoreceptor gets a chunk of charge, simulating light hitting the eye

In [ ]:
decay_factor = 0.8
threshold = 5
time_steps = 6

external_input = {11139: 10}

#### Running the simulation

In [ ]:
for t in range(time_steps):
    print(f"--- time step {t} ---")

    # apply external input only at the very first tick
    if t == 0:
        for nid, amount in external_input.items():
            print("external applied to", nid)
            charge[nid] += amount

    # figure out who fires this tick, based on charge BEFORE any resets happen
    fired = [nid for nid in neuron_ids if charge[nid] >= threshold]

    # apply the leak to everyone
    for nid in neuron_ids:
        charge[nid] *= decay_factor

    # firing neurons dump their charge to neighbors, then reset to 0
    for pre, post, weight in edges:
        if pre in fired:
            print(f"{pre} fires away to {post}")
            charge[post] += weight

    for nid in fired:
        charge[nid] = 0

    print({nid: round(c, 2) for nid, c in charge.items()})

> while the others slowly go to rest, 26947 and 29782 pass the threshold and fire, but they don't have any edges connecting to anyone else, so they fire once and go straight to rest

### Conclusion

what was learned from this experiment:
- `ol_sensory` to `vnc_motor` means the charge has to travel quite far
- for that to happen, weights would need to change so the right paths get reinforced, some rule that adjusts weights based on simulation runs, down the line
- bottom line: the fly is dead, so this file is a starting point, not a trained fly

from here, the decision can be made to expand what's currently built, to get a charge from `ol_sensory` to `vnc_motor`, or circle back to that "some rule" (learning) first

to expand what's built: collect neighbours of 26947, 29782, add them to `neuron_ids`/`edges`, then repeat until reaching `vnc_motor`
- nothing about the simulation logic changes, maybe the parameters, only `neuron_ids` and `edges` grow
- note: graph traversal (dfs/bfs) exists for exactly this problem, finding a `vnc_motor` neuron as the target, while finding the path that lets charge travel there

### Side track: is there a right-side counterpart to `11139`?

while looking at the ol_sensory with bodyId `11139` (instance `R1-R6_L`), the question came up whether it has a right-side counterpart. checking `somaSide` turned up something bigger than the question, so it's kept here at the end, away from the LIF model

In [ ]:
traced = df[df['status'] == 'Traced']
print(traced[traced['type'] == 'R1-R6']['somaSide'].value_counts())

> R1-R6 is different, it's a type shared by photoreceptors across the whole compound eye
>
> the right-side counterpart to 11139 specifically isn't a single neuron that can be pointed to, the way DNp01(GF)_L was for DNp01(GF)_R
>
> also for some reason only 13 neurons are returned? this will need investigating:

In [ ]:
print(traced[traced['type'] == 'R1-R6']['somaSide'].value_counts(dropna=False))
print(traced[traced['superclass'] == 'ol_sensory']['somaSide'].value_counts(dropna=False))

> 1381 out of 1394 R1-R6 neurons have no somaSide recorded
>
> also the gap is structural, not specific to R1-R6. The whole ol_sensory superclass shows the same pattern, 4078 of roughly 4114 missing somaSide
>
> next, check if that pattern holds across the other superclasses too, or whether ol_sensory is uniquely bad:

In [ ]:
# gives the fraction missing per superclass in one pass
print(traced.groupby('superclass')['somaSide'].apply(lambda s: s.isna().mean()).sort_values(ascending=False))

> Every sensory superclass, cb_sensory, vnc_sensory, ol_sensory, ENS, every sensory_* variant, sits between 97.9% and 100% missing
>
> Every other superclass, intrinsic, motor, descending, ascending, efferent, endocrine, visual_projection, sits at essentially 0% missing
>
> somaSide was, for practical purposes, only ever annotated for non-sensory neurons for some reason.